# Snap-to-Sell — Google Colab Demo (live, with real APIs)

Runs the pipeline **for real** (Gemini recognition + Open Food Facts / Open Products Facts
retrieval) in Colab, which has open network access. Steps: install deps → load the code →
paste your Gemini key → run.

> Runtime → *Run all* after uploading the code (Step 2) and pasting your key (Step 3).

### Step 1 — install dependencies

In [ ]:
!pip install -q requests pillow opencv-python-headless imagehash pandas matplotlib python-dotenv rembg onnxruntime

### Step 2 — load the code

Two options:
- **Upload a zip** (easiest): run the cell, then choose `snap-to-sell.zip` (provided alongside
  this notebook, or zip your local `snap-to-sell/` folder).
- **Git clone**: if you've pushed the repo, uncomment the clone line instead.

In [ ]:
import os, sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

# Already inside the repo (local Jupyter run from snap-to-sell/)? Just fix imports — no zip, no upload.
if os.path.isdir("src/snap_to_sell"):
    pass
elif os.path.isdir("snap-to-sell/src/snap_to_sell"):
    os.chdir("snap-to-sell")
elif _in_colab():
    import zipfile, io
    from google.colab import files
    up = files.upload()                      # choose snap-to-sell.zip
    fn = next(iter(up))
    zipfile.ZipFile(io.BytesIO(up[fn])).extractall(".")
    os.chdir("snap-to-sell")
    # --- OR clone from GitHub instead of uploading a zip ---
    # !git clone https://github.com/<your-org>/MIA5100_group_project.git
    # !cp -r MIA5100_group_project/snap-to-sell . && cd snap-to-sell
else:
    raise RuntimeError("Run this notebook from inside the snap-to-sell/ folder, "
                       "or provide snap-to-sell.zip in Colab.")

sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print("test images:", [f for f in os.listdir("data/testset") if f.endswith(('.jpg', '.jpeg'))])

### Step 3 — configuration

Set your provider, model, key and strict-mode here. The key is entered via `getpass` (masked) if
you leave it blank below. Re-run this cell to change any setting.

In [ ]:
import os, getpass

# Blank = use whatever is in your .env (recommended). Fill a field ONLY to override .env for this run.
SNAP_LLM_PROVIDER     = ""    # "openai" | "anthropic" | "gemini" | ""  -> blank uses .env / auto-detect
SNAP_OPENAI_MODEL     = ""    # e.g. "gpt-4o-mini"; blank -> .env / default
SNAP_RETRIEVE_BACKEND = ""    # "off" (Open Food Facts) | "shopping" (Serper); blank -> .env
OPENAI_API_KEY        = ""    # paste to override .env; blank -> use .env
SERPER_API_KEY        = ""    # blank -> use .env  (needed for backend="shopping")
SNAP_STRICT_PROVIDER  = None  # True / False to override; None -> use .env

# --- only non-empty cell values override .env ---
for _k, _v in {
    "SNAP_LLM_PROVIDER": SNAP_LLM_PROVIDER,
    "SNAP_OPENAI_MODEL": SNAP_OPENAI_MODEL,
    "SNAP_RETRIEVE_BACKEND": SNAP_RETRIEVE_BACKEND,
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "SERPER_API_KEY": SERPER_API_KEY,
}.items():
    if _v:
        os.environ[_k] = _v
if SNAP_STRICT_PROVIDER is not None:
    os.environ["SNAP_STRICT_PROVIDER"] = "1" if SNAP_STRICT_PROVIDER else ""

# config auto-loads the nearest .env (override=False), so keys in .env are picked up here.
from src.snap_to_sell import config
config.refresh()

# Prompt for a key ONLY if the chosen provider is OpenAI and no key came from the cell or .env.
_provider = (os.environ.get("SNAP_LLM_PROVIDER") or config.LLM_PROVIDER or "").lower()
if _provider == "openai" and not config.OPENAI_API_KEY:
    try:
        _k = getpass.getpass("OPENAI_API_KEY (blank = offline): ")
    except Exception:
        _k = ""
    if _k:
        os.environ["OPENAI_API_KEY"] = _k
        config.refresh()

# Convenience: with the shopping backend, show the clean retailer image (unless you set SNAP_IMAGE_MODE).
if config.RETRIEVE_BACKEND == "shopping" and not os.environ.get("SNAP_IMAGE_MODE"):
    os.environ["SNAP_IMAGE_MODE"] = "swap"
    config.refresh()

print("retrieval:", config.RETRIEVE_BACKEND)
print("provider :", config.active_provider() or "offline (no key)")
print("model    :", config.OPENAI_MODEL)
print("strict   :", config.STRICT_PROVIDER)

### Step 4 — run the pipeline on a product (real recognition + retrieval)

In [ ]:
%matplotlib inline
import json
from dataclasses import asdict
import matplotlib.pyplot as plt
from PIL import Image
from src.snap_to_sell.pipeline import run

SAMPLE = "data/testset/duracell_aa.jpg"
plt.figure(figsize=(3.2, 4.2)); plt.imshow(Image.open(SAMPLE)); plt.axis("off"); plt.title("Input"); plt.show()

result = run(SAMPLE)
print(json.dumps(asdict(result), indent=2, default=str))

### Step 5 — age-restricted products (guardrail)

In [ ]:
for name, path in [("Marlboro", "data/testset/marlboro.jpg"),
                   ("Flying Horse (Chinese pack)", "data/testset/flyinghorse.jpg")]:
    plt.figure(figsize=(2.8, 3.6)); plt.imshow(Image.open(path)); plt.axis("off"); plt.title(name); plt.show()
    v = run(path)
    print(f"{name}: compliance={v.compliance_status} | age_restricted={v.listing.flags.age_restricted}")
    print("  title:", v.listing.title, "\n")

### Step 6 — evaluation over the test set

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("harness", "eval/harness.py")
harness = importlib.util.module_from_spec(spec); spec.loader.exec_module(harness)
harness.main()

### Step 7 — product listings (HTML grid)

Results as e-commerce product cards. An example grid over the bundled samples first, then upload
your own — the grid renders after all images are processed.

In [ ]:
import os, base64, mimetypes, html as _html
from src.snap_to_sell.pipeline import run
from IPython.display import HTML, display

def _img_src(ref):
    if str(ref).startswith("http"):
        return ref
    mime = mimetypes.guess_type(ref)[0] or "image/jpeg"
    with open(ref, "rb") as f:
        return f"data:{mime};base64," + base64.b64encode(f.read()).decode()

_CSS = """<style>
.snap-grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(230px,1fr));gap:18px;padding:10px 2px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;}
.snap-card{background:#fff;border:1px solid #ececf0;border-radius:16px;overflow:hidden;box-shadow:0 2px 10px rgba(20,20,40,.06);display:flex;flex-direction:column;transition:transform .15s ease,box-shadow .15s ease;}
.snap-card:hover{transform:translateY(-3px);box-shadow:0 10px 26px rgba(20,20,40,.13);}
.snap-imgwrap{position:relative;background:#f6f6f8;aspect-ratio:1/1;display:flex;align-items:center;justify-content:center;}
.snap-imgwrap img{width:100%;height:100%;object-fit:contain;}
.snap-swap{position:absolute;top:8px;left:8px;background:rgba(0,0,0,.62);color:#fff;font-size:10px;padding:3px 8px;border-radius:20px;}
.snap-body{padding:12px 14px 14px;display:flex;flex-direction:column;gap:6px;flex:1;}
.snap-cat{font-size:11px;letter-spacing:.06em;text-transform:uppercase;color:#8a8a99;font-weight:600;}
.snap-title{font-size:14px;font-weight:650;color:#1c1c28;line-height:1.3;min-height:36px;}
.snap-price{font-size:18px;font-weight:750;color:#0a7d3c;}
.snap-desc{font-size:12px;color:#5b5b6b;line-height:1.4;flex:1;}
.snap-badges{display:flex;flex-wrap:wrap;gap:6px;margin-top:6px;}
.snap-badge{font-size:10.5px;font-weight:600;padding:3px 9px;border-radius:20px;}
.b-ok{background:#e6f7ee;color:#0a7d3c;}.b-age{background:#fdecea;color:#c0392b;}
.b-review{background:#fff4e0;color:#b9770a;}.b-warn{background:#eef0ff;color:#4a4ad0;}
.snap-log{margin-top:8px;font-size:11px;}
.snap-log summary{cursor:pointer;color:#4a4ad0;font-weight:600;}
.snap-log pre{max-height:240px;overflow:auto;background:#f7f7fb;border:1px solid #ececf0;border-radius:8px;padding:8px;font-size:10px;line-height:1.35;white-space:pre-wrap;}
</style>"""

def _log_html(path):
    lp = os.path.join(os.path.dirname(path), os.path.splitext(os.path.basename(path))[0] + "-log.txt")
    if not os.path.exists(lp):
        return ""
    try:
        txt = open(lp, encoding="utf-8").read()
    except Exception:
        return ""
    return ('<details class="snap-log"><summary>view processing log (' + _html.escape(os.path.basename(lp))
            + ')</summary><pre>' + _html.escape(txt) + '</pre></details>')

def _card(path, v):
    l = v.listing; idy = l.identity
    ref = l.image_url if str(l.image_url).startswith("http") else path
    price = f"{l.price.currency} {l.price.point:.2f}" if l.price.point else "Price on request"
    badges = ""
    if l.flags.age_restricted: badges += '<span class="snap-badge b-age">18+ age-restricted</span>'
    if v.compliance_status == "flagged": badges += '<span class="snap-badge b-review">needs review</span>'
    if l.flags.low_confidence: badges += '<span class="snap-badge b-warn">low confidence</span>'
    if not badges: badges = '<span class="snap-badge b-ok">ready to publish</span>'
    swap = ('<span class="snap-swap">catalogue image</span>' if l.flags.image_swapped
            else '<span class="snap-swap">cleaned photo</span>' if l.flags.image_enhanced else "")
    return ('<div class="snap-card"><div class="snap-imgwrap"><img src="' + _img_src(ref) + '"/>' + swap + '</div>'
            '<div class="snap-body"><div class="snap-cat">' + _html.escape(idy.category or "-") + '</div>'
            '<div class="snap-title">' + _html.escape(l.title or "Untitled") + '</div>'
            '<div class="snap-price">' + _html.escape(price) + '</div>'
            '<div class="snap-desc">' + _html.escape(l.description or "") + '</div>'
            '<div class="snap-badges">' + badges + '</div>' + _log_html(path) + '</div></div>')

def render_grid(results):
    """results: list of (image_path, ValidatedListing) — renders all cards in a grid."""
    if not results:
        display(HTML("<p>No images processed.</p>")); return
    cards = "".join(_card(p, v) for p, v in results)
    display(HTML(_CSS + '<div class="snap-grid">' + cards + '</div>'))

def process(paths):
    out = []
    for p in paths:
        try:
            out.append((p, run(p)))
        except Exception as e:
            print("skip", os.path.basename(p), "->", e)
    return out

Example grid on the bundled sample products:

In [ ]:
render_grid(process(["data/testset/duracell_aa.jpg",
                     "data/testset/marlboro.jpg",
                     "data/testset/flyinghorse.jpg"]))

### Upload your own

In [ ]:
# Works in Colab (google.colab.files) and locally (ipywidgets fallback) — no zip needed.
os.makedirs("data/uploads", exist_ok=True)

def _save_uploads(items):
    paths = []
    for name, content in items:
        p = os.path.join("data/uploads", name)
        with open(p, "wb") as w:
            w.write(bytes(content))
        paths.append(p)
    return paths

try:
    from google.colab import files
    uploaded = files.upload()          # choose one or more images
    paths = _save_uploads(list(uploaded.items()))
    if paths:
        render_grid(process(paths))
except ImportError:
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output
        uploader = widgets.FileUpload(accept="image/*", multiple=True)
        out = widgets.Output()
        def _on_upload(change):
            with out:
                clear_output()
                vals = uploader.value
                fs = list(vals.values()) if isinstance(vals, dict) else list(vals)
                items = [((f.get("name") or f.get("metadata", {}).get("name", "upload.jpg")), f["content"]) for f in fs]
                paths = _save_uploads(items)
                if paths:
                    render_grid(process(paths))
        uploader.observe(_on_upload, names="value")
        print("Upload one or more product photos:")
        display(uploader, out)
    except ImportError:
        print("ipywidgets not installed -> pip install ipywidgets")

### Notes
- Update `data/testset/labels.csv` with real observed prices before trusting the price metric.
- If a stage errors, the pipeline falls back gracefully (see the offline demo notebook
  `snap_demo.ipynb`). Architecture: `docs/final_report/figures/architecture_impl.png`.